# FORML binary classification policy

Label, probability, pairwise verification, reports, and replay through the public API.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import importlib.util
import sys

from forml import verify


_notebook_directory = Path.cwd().resolve()
_repository_root = next(
    candidate
    for candidate in (_notebook_directory, *_notebook_directory.parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "dsl").is_dir()
)
_repository_root_text = str(_repository_root)
if _repository_root_text not in sys.path:
    sys.path.insert(0, _repository_root_text)

_demo_path = _repository_root / "demo" / "binary_classification_policy.py"
_demo_spec = importlib.util.spec_from_file_location(
    "_forml_binary_classification_policy_demo",
    _demo_path,
)
if _demo_spec is None or _demo_spec.loader is None:
    raise RuntimeError(f"Cannot load the binary classification demo from {_demo_path}")
_demo_module = importlib.util.module_from_spec(_demo_spec)
sys.modules[_demo_spec.name] = _demo_module
_demo_spec.loader.exec_module(_demo_module)
build_demo_artifacts = _demo_module.build_demo_artifacts

In [ ]:
workspace = TemporaryDirectory(prefix="forml-binary-notebook-")
workspace_path = Path(workspace.name)
model_path, dataset_path = build_demo_artifacts(workspace_path)
specification_path = _repository_root / "demo" / "binary_classification_policy.forml"
session = verify(specification_path, model=model_path, dataset=dataset_path)
session


In [ ]:
session.to_dataframe()


In [ ]:
counterexample = session.first_counterexample
assert counterexample is not None
replay = counterexample.replay()
replay.to_dataframe()


In [ ]:
artifacts = session.write_artifacts(workspace_path / "reports")
artifacts
